# UbuntuCare — Docker-free Colab test environment

Run all cells in order in **Google Colab**. PostgreSQL, Redis, Keycloak, Qdrant, Neo4j, MinIO, Ollama, the application APIs, speech services, RAG, Celery, and frontend run directly inside the Colab VM. No Docker, local computer, or existing website URL is needed.

First setup downloads dependencies/models and builds the frontend and MinIO; allow substantial startup time. Use a runtime with at least 10 GiB RAM and 25 GiB free disk. Speech uses tiny Whisper on CPU and the agents use Qwen 2.5 1.5B for resource-limited functional testing, not clinical validation.

The **temporary URL cell** displays the link only after admin and doctor login succeed. Doctor: `john.mwamba@uh.gov.zm`; admin: `admin@ubuntucare.gov.zm`; demo password: `UbuntuCare2024!`. Anyone with this public link and these shared credentials can access the demo. **Synthetic data only.** Data and URL disappear when Colab terminates the runtime.

Scope: all existing application service entrypoints start. Laboratory/pharmacy/imaging are currently health-only shells; launching them cannot implement missing features. External SMS, email, WhatsApp and clinical integrations need separate provider configuration. The production monitoring/management stack (ELK, Grafana, Prometheus, Portainer, Nginx) and secondary Weaviate are not run inside this demo. The launcher reports actual process status rather than claiming these features are complete.

In [ ]:
%pip -q install requests
import requests

## 0. Clone and prepare the Colab runtime

Run the setup, data/application startup, and temporary URL cells in order. The install step writes progress to `/content/ubuntucare-native/logs/`. On failure inspect the named log privately; setup logs contain credentials. Rerun startup cells using the same `runtime` object; use a fresh runtime if another launcher owns the ports. This installs packages only inside the Colab VM.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time
from datetime import datetime, timezone
from uuid import uuid4
import requests

if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('Open this notebook in Google Colab. This launcher does not run services on your computer.')
REPO_URL = 'https://github.com/rager1904/beverly-eye.git'
BRANCH = 'main'
if 'REPO_DIR' not in globals():
    REPO_DIR = Path('/content/ubuntucare-colab-source')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / 'scripts/colab/native_runtime.py').exists():
    raise RuntimeError('This checkout predates the native launcher. Open the current GitHub notebook in a fresh Colab runtime.')
sys.path.insert(0, str(REPO_DIR / 'scripts/colab'))
from native_runtime import ColabRuntime
if 'runtime' not in globals():
    runtime = ColabRuntime()
LOCAL_LAUNCH = True  # local to the Colab VM, never your own computer
RUN_WRITE_TESTS = False
STOP_DEMO = False
DEMO_LOGIN_EMAIL = 'admin@ubuntucare.gov.zm'
DEMO_DOCTOR_EMAIL = 'john.mwamba@uh.gov.zm'
DEMO_LOGIN_PASSWORD = 'UbuntuCare2024!'
HOSPITAL_ID = '00000000-0000-4000-8000-000000000001'

def show_frontend_url(url, temporary=False):
    from html import escape
    from urllib.parse import urlparse
    parsed_url = urlparse(url)
    if parsed_url.scheme != 'https' or not parsed_url.hostname or parsed_url.username or parsed_url.password or parsed_url.query or parsed_url.fragment or parsed_url.path not in {'', '/'}:
        raise ValueError('Frontend URL must be an HTTPS origin without credentials, path, query, or fragment.')
    url = url.rstrip('/')
    label = 'Your temporary frontend URL' if temporary else 'Your connected frontend URL'
    print(f'{label}: {url}')
    print(f'Open login: {url}/login')
    if temporary:
        print('Temporary link: available only while the runtime, frontend, and tunnel remain running.')
    else:
        print('Existing deployment link; this notebook did not create a new tunnel.')
    try:
        from IPython.display import HTML, display
    except ImportError:
        return  # Plain-text links remain available outside a notebook.
    safe_url = escape(url, quote=True)
    display(HTML(f'<div style="border:2px solid #168777;padding:16px;border-radius:8px">'
                 f'<strong>{label}</strong><p><a href="{safe_url}/login" target="_blank" rel="noopener noreferrer">'
                 f'{safe_url}/login</a></p></div>'))

print('Source ready:', REPO_DIR)

In [ ]:
# Install native Linux services and an isolated Python environment; build the frontend.
# First run may take 15–30+ minutes depending on Colab and download speeds.
runtime.install()

In [ ]:
# Everything below starts inside this Colab VM, without Docker or a remote URL.
runtime.start_data()
runtime.start_apps()
runtime.status()

In [ ]:
# TEMPORARY FRONTEND URL — no URL input is required.
# Refuses to publish until both demo accounts can authenticate through the frontend.
FRONTEND_URL = runtime.expose()
BASE_URL = FRONTEND_URL
show_frontend_url(FRONTEND_URL, temporary=True)
print('Doctor email:', DEMO_DOCTOR_EMAIL)
print('Admin email:', DEMO_LOGIN_EMAIL)
print('Demo password:', DEMO_LOGIN_PASSWORD)
print('Use the email/password form. Keep this runtime running; synthetic demo data only.')

In [ ]:
# Authenticate against the Colab frontend itself; no external URL or credentials prompt.
session = requests.Session()
session.headers.update({'Accept': 'application/json'})
API = f'{FRONTEND_URL}/api/v1'
login_response = session.post(f'{API}/auth/login', json={'email': DEMO_LOGIN_EMAIL, 'password': DEMO_LOGIN_PASSWORD}, timeout=45, allow_redirects=False)
if login_response.status_code != 200:
    raise RuntimeError(f'Colab demo login failed: HTTP {login_response.status_code}. Check runtime.status() and gateway/keycloak logs.')
ACCESS_TOKEN = login_response.json().get('access_token')
if not isinstance(ACCESS_TOKEN, str) or not ACCESS_TOKEN:
    raise RuntimeError('Login returned no access token.')
login_response = None
session.headers['Authorization'] = f'Bearer {ACCESS_TOKEN}'
session.headers['X-Hospital-ID'] = HOSPITAL_ID
print('Authenticated Colab demo:', FRONTEND_URL)

## 1. Gateway health (read-only)

This request does not need a token. A `healthy` result means the gateway can reach its database and Redis checks; other dependencies may still need separate attention.

In [ ]:
health_url = 'http://127.0.0.1:8000/health' if LOCAL_LAUNCH else f'{BASE_URL}/health'
health_response = requests.get(health_url, timeout=30, allow_redirects=False)
print('GET /health:', health_response.status_code)
if health_response.status_code == 404 and not LOCAL_LAUNCH:
    print('Gateway health is not proxied by this frontend URL; continuing with authenticated API checks.')
else:
    if health_response.status_code != 200:
        raise RuntimeError(f'Gateway health failed: HTTP {health_response.status_code}')
    health_response.raise_for_status()
    health = health_response.json()
    print('Gateway status:', health.get('status'))
    print('Database:', health.get('checks', {}).get('database', {}).get('status'))
    print('Redis:', health.get('checks', {}).get('redis', {}).get('status'))
    if health.get('status') != 'healthy':
        raise RuntimeError('Gateway reports unhealthy dependencies.')

## 2. Authenticated read-only checks

These checks verify the seeded departments, staff directory, patient flow, patient search, and note API through the public Colab frontend. No real patient records are printed. AI service readiness is checked separately by the native launcher; process startup alone does not establish model accuracy.

In [ ]:
def get_json(path, params=None):
    response = session.get(f'{API}{path}', params=params, timeout=30, allow_redirects=False)
    print(f'GET {path}:', response.status_code)
    if response.status_code != 200:
        raise RuntimeError(f'GET {path} failed: HTTP {response.status_code}. Check URL, tenant, role, and token expiry.')
    return response.json()

departments_data = get_json('/hospital/departments')
departments = departments_data if isinstance(departments_data, list) else departments_data.get('items', [])
active_department_names = {d['name'] for d in departments if d.get('is_active', True) and isinstance(d.get('name'), str)}
print('Active departments:', sorted(active_department_names))
expected_departments = {'General', 'Eye', 'Dental'}
assert expected_departments.issubset(active_department_names), f'Missing required departments; expected {sorted(expected_departments)}, got {sorted(active_department_names)}'

staff_data = get_json('/hospital/staff', params={'per_page': 100})
staff_items = staff_data.get('items', []) if isinstance(staff_data, dict) else staff_data
print('Directory staff records available:', len(staff_items))

destinations = get_json('/patient-flow/destinations')
service_points = {item.get('code') for item in destinations.get('service_points', [])}
print('Core service destinations present:', sorted({'general', 'eye', 'dental'} & service_points))
assert {'general', 'eye', 'dental'}.issubset(service_points), 'General/Eye/Dental patient-flow destinations are missing'

# A unique synthetic query checks patient search without listing or displaying real patients.
search_marker = f'COLAB-NONEXISTENT-{uuid4().hex[:10]}'
patient_search = get_json('/emr/patients', params={'query': search_marker, 'per_page': 5})
print('Synthetic patient search completed; matches:', patient_search.get('total', 0))

notes_data = get_json('/emr/notes', params={'patient_id': str(uuid4()), 'page': 1, 'per_page': 1})
print('Clinical-note API accessible; total notes:', notes_data.get('total', 'not reported'))
print('Read-only checks passed.')

## 3. Optional write test — disabled by default

Only continue in the temporary Colab database or a **test/staging environment**. Set `RUN_WRITE_TESTS = True` in the configuration cell, then run this cell. It creates one clearly labelled synthetic patient, an Eye visit with NHIMA/review/paid test values, and a draft eye SOAP note. These records remain in the database; the notebook does not delete records. Do not use real names, NRCs, phone numbers, or clinical information.

In [ ]:
if not RUN_WRITE_TESTS:
    print('Skipped: set RUN_WRITE_TESTS = True only after confirming this is a test/staging deployment.')
else:
    marker = uuid4().hex[:8].upper()
    eye_department = next((d for d in departments if d.get('name') == 'Eye' and d.get('is_active', True)), None)
    if eye_department is None:
        raise RuntimeError('Run the read-only checks first: active Eye department is required before writing data.')
    patient_payload = {
        'first_name': 'ColabTest',
        'last_name': f'Synthetic{marker}',
        'date_of_birth': '1990-01-01',
        'gender': 'other',
        'phone': '+260000000000',
        'notes': 'SYNTHETIC COLAB SMOKE TEST DATA — not a real patient.',
    }
    patient_response = session.post(f'{API}/emr/patients', json=patient_payload, timeout=30, allow_redirects=False)
    print('POST /emr/patients:', patient_response.status_code)
    if patient_response.status_code != 201:
        raise RuntimeError('Creating patient failed: HTTP ' + str(patient_response.status_code) + '. Earlier synthetic records may remain; do not rerun blindly.')
    patient_id = patient_response.json()['id']

    eye_department = next((d for d in departments if d.get('name') == 'Eye' and d.get('is_active', True)), None)
    journey_payload = {
        'patient_id': patient_id,
        'priority': 'routine',
        'payment_scheme': 'nhima',
        'visit_type': 'review',
        'payment_status': 'paid',
        'scheme_number': f'COLAB-{marker}',
        'chief_complaint': 'SYNTHETIC TEST DATA ONLY — routine eye review',
        'step_type': 'consultation',
        'service_point': 'eye',
        'department_id': eye_department.get('id') if eye_department else None,
        'department_name': 'Eye',
    }
    journey_response = session.post(f'{API}/patient-flow/journeys', json=journey_payload, timeout=30, allow_redirects=False)
    print('POST /patient-flow/journeys:', journey_response.status_code)
    if journey_response.status_code != 201:
        raise RuntimeError('Creating journey failed: HTTP ' + str(journey_response.status_code) + '. Earlier synthetic records may remain; do not rerun blindly.')
    journey = journey_response.json()
    print('Created synthetic journey:', journey.get('journey_number'), '|', journey.get('visit_type'), '|', journey.get('payment_status'))

    note_payload = {
        'patient_id': patient_id,
        'note_type': 'SOAP',
        'title': f'Colab Smoke Test - Eye Note {marker}',
        'subjective': 'SYNTHETIC TEST DATA ONLY — no real patient symptoms.',
        'objective': 'SYNTHETIC TEST DATA ONLY — no examination performed.',
        'assessment': 'Test record only; not a clinical diagnosis.',
        'plan': 'Test record only; no treatment.',
        'authored_by': 'Colab Smoke Test',
        'authored_datetime': datetime.now(timezone.utc).isoformat(),
        'status': 'draft',
    }
    note_response = session.post(f'{API}/emr/notes', json=note_payload, timeout=30, allow_redirects=False)
    print('POST /emr/notes:', note_response.status_code)
    if note_response.status_code != 201:
        raise RuntimeError('Creating note failed: HTTP ' + str(note_response.status_code) + '. Earlier synthetic records may remain; do not rerun blindly.')
    print('Created synthetic draft note:', note_response.json().get('id'))
    print('Write smoke test passed. Synthetic records were left in the test database.')

## 4. Stop the Colab demo (opt-in)

Set `STOP_DEMO = True` in this cell and run it when finished. It terminates only the subprocesses tracked by this launcher, closes the public tunnel, and retains runtime files. Nothing is stopped on your own computer.

In [ ]:
STOP_DEMO = False  # Change to True when finished, then rerun this cell.
if STOP_DEMO:
    runtime.stop()
    if 'session' in globals():
        session.close()
        session.headers.pop('Authorization', None)
    globals().pop('ACCESS_TOKEN', None)
else:
    print('Demo remains running in Colab. Set STOP_DEMO = True here to close the URL and stop services.')

## Troubleshooting and limits

- **Install/startup failure:** the exception names a private log in `/content/ubuntucare-native/logs/`. Do not share raw setup logs or tokens.
- **Invalid login:** run `runtime.status()` and inspect gateway/Keycloak logs. The URL cell verifies both demo logins before displaying the link.
- **AI readiness:** models download from their publishers. A tiny CPU speech model and 1.5B LLM trade quality for memory usage; this is not clinical evaluation. TTS currently depends on external Edge/Google services and can fall back to a tone; a live service is not proof of successful speech generation.
- **Memory/disk limits:** upgrade the runtime allocation if needed. Colab is temporary test infrastructure, not commercial hosting; it may disconnect or terminate processes.
- **Missing features:** health-only service shells and unregistered queue tasks remain repository implementation gaps; see the service report and README.
- **No existing URL is needed:** the launcher generates a new frontend tunnel inside Colab. Do not reuse the previous computer-hosted tunnel.